In [ ]:
from reedsolo import RSCodec, rs_calc_syndromes
import os
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from rs.channels import qsc_erasure_channel
from rs.dataset_gen import bytes_to_bits, get_zero_mask, RSPositionDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
class PositionPredictor(nn.Module):
    def __init__(self, use_batchnorm=False, dropout_rate=0.0):
        super().__init__()

        layers = []

        layers.append(nn.Linear(511, 512))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(512))
        layers.append(nn.ReLU())
        if dropout_rate > 0:
            layers.append(nn.Dropout(dropout_rate))
        
        for _ in range(3):
            layers.append(nn.Linear(512, 512))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(512))
            layers.append(nn.ReLU())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
        
        layers.append(nn.Linear(512, 255))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)